# 📊 Data Preprocessing & Feature Engineering Project

## 🧠 Objective
Handle missing values and outliers in a healthcare dataset using multiple techniques.

## 📁 Dataset
Includes:
- Demographics (Age, Gender, Region)
- Medical data (BMI, BP, Cholesterol, Glucose)
- Target (Disease Risk)

# Part A: Handling Missing Values

## 🔹 Step 1: Dataset Generation

Generate synthetic healthcare data with:
- Missing values
- Outliers
- Realistic distributions

In [39]:
import numpy as np
import pandas as pd

np.random.seed(42)
n = 500

data = pd.DataFrame({
    "patient_id": np.arange(1, n+1),
    "age": np.random.randint(18, 80, n),
    "gender": np.random.choice(["Male", "Female"], n),
    "region": np.random.choice(["North", "South", "East", "West"], n),
    "bmi": np.random.normal(25, 5, n),
    "blood_pressure": np.random.normal(120, 15, n),
    "cholesterol": np.random.normal(200, 40, n),
    "glucose": np.random.normal(100, 30, n),
})

data["disease_risk"] = np.random.choice([0, 1], n)

for col in ["age", "gender", "region", "bmi", "cholesterol", "glucose"]:
    data.loc[data.sample(frac=0.1).index, col] = np.nan

outliers = np.random.choice(data.index, 20)
data.loc[outliers, "bmi"] *= 2
data.loc[outliers, "blood_pressure"] *= 1.8
data.loc[outliers, "cholesterol"] *= 2
data.loc[outliers, "glucose"] *= 2.5

data.to_csv("health_data.csv", index=False)
data.head()

,patient_id,age,gender,region,bmi,blood_pressure,cholesterol,glucose,disease_risk
0,1,56.0,NaN,South,19.543048,117.996302,215.179282,105.768789,0
1,2,69.0,NaN,South,20.914993,136.287995,272.460193,95.553314,1
2,3,46.0,NaN,West,32.548857,88.486217,NaN,73.645515,0
3,4,32.0,Male,South,27.010172,164.066293,166.839323,57.623927,1
4,5,60.0,Male,South,NaN,106.932994,149.346380,47.684557,1


## 🔹 Step 2: Load Dataset

In [40]:
data = pd.read_csv("health_data.csv")
data.head()

,patient_id,age,gender,region,bmi,blood_pressure,cholesterol,glucose,disease_risk
0,1,56.0,NaN,South,19.543048,117.996302,215.179282,105.768789,0
1,2,69.0,NaN,South,20.914993,136.287995,272.460193,95.553314,1
2,3,46.0,NaN,West,32.548857,88.486217,NaN,73.645515,0
3,4,32.0,Male,South,27.010172,164.066293,166.839323,57.623927,1
4,5,60.0,Male,South,NaN,106.932994,149.346380,47.684557,1


## 🔹 Missing Value Analysis
Check total missing values.

In [41]:
data.isnull().sum()

patient_id         0
age               50
gender            50
region            50
bmi               50
blood_pressure     0
cholesterol       50
glucose           50
disease_risk       0
dtype: int64

check total missing value percentage

In [42]:
data.isnull().mean()*100

patient_id         0.0
age               10.0
gender            10.0
region            10.0
bmi               10.0
blood_pressure     0.0
cholesterol       10.0
glucose           10.0
disease_risk       0.0
dtype: float64

## 🔹 Simple Imputation (Numerical)

In this step, missing values in numerical columns are replaced using the mean value.

This method is simple and fast but may slightly distort data distribution.

In [43]:
data_simple = data.copy()

data_simple["bmi"] = data_simple["bmi"].fillna(data_simple["bmi"].mean())
data_simple["cholesterol"] = data_simple["cholesterol"].fillna(data_simple["cholesterol"].mean())

print("✅ Missing values after Simple Imputation:")
print(data_simple[["bmi", "cholesterol"]].isnull().sum())

data_simple.head()

✅ Missing values after Simple Imputation:
bmi            0
cholesterol    0
dtype: int64


,patient_id,age,gender,region,bmi,blood_pressure,cholesterol,glucose,disease_risk
0,1,56.0,NaN,South,19.543048,117.996302,215.179282,105.768789,0
1,2,69.0,NaN,South,20.914993,136.287995,272.460193,95.553314,1
2,3,46.0,NaN,West,32.548857,88.486217,209.286945,73.645515,0
3,4,32.0,Male,South,27.010172,164.066293,166.839323,57.623927,1
4,5,60.0,Male,South,26.008593,106.932994,149.346380,47.684557,1


## 🔹 Most Frequent Imputation (Categorical)

Missing values in categorical columns are replaced with the most frequent value (mode).

This method works well for categorical data.

In [44]:
data_mode = data.copy()

data_mode["gender"] = data_mode["gender"].fillna(data_mode["gender"].mode()[0])
data_mode["region"] = data_mode["region"].fillna(data_mode["region"].mode()[0])

print("✅ Missing values after Mode Imputation:")
print(data_mode[["gender", "region"]].isnull().sum())

data_mode.head()

✅ Missing values after Mode Imputation:
gender    0
region    0
dtype: int64


,patient_id,age,gender,region,bmi,blood_pressure,cholesterol,glucose,disease_risk
0,1,56.0,Male,South,19.543048,117.996302,215.179282,105.768789,0
1,2,69.0,Male,South,20.914993,136.287995,272.460193,95.553314,1
2,3,46.0,Male,West,32.548857,88.486217,NaN,73.645515,0
3,4,32.0,Male,South,27.010172,164.066293,166.839323,57.623927,1
4,5,60.0,Male,South,NaN,106.932994,149.346380,47.684557,1


## 🔹 Missing Indicator + Random Sampling

A new column is created to indicate missing values.

Missing values are filled using random sampling from existing values, preserving the data distribution.

In [45]:
data_random = data.copy()

data_random["bmi_missing"] = data_random["bmi"].isnull().astype(int)

data_random["bmi"] = data_random["bmi"].apply(
    lambda x: data_random["bmi"].dropna().sample(1).values[0] if pd.isnull(x) else x
)

print("✅ Missing values after Random Sampling:")
print(data_random["bmi"].isnull().sum())

data_random.head()

✅ Missing values after Random Sampling:
0


,patient_id,age,gender,region,bmi,blood_pressure,cholesterol,glucose,disease_risk,bmi_missing
0,1,56.0,NaN,South,19.543048,117.996302,215.179282,105.768789,0,0
1,2,69.0,NaN,South,20.914993,136.287995,272.460193,95.553314,1,0
2,3,46.0,NaN,West,32.548857,88.486217,NaN,73.645515,0,0
3,4,32.0,Male,South,27.010172,164.066293,166.839323,57.623927,1,0
4,5,60.0,Male,South,18.730959,106.932994,149.346380,47.684557,1,1


## 🔹 KNN Imputation

K-Nearest Neighbors imputes missing values based on similarity between data points.

It considers relationships between variables.

In [46]:
from sklearn.impute import KNNImputer

data_knn = data.copy()
num_cols = ["age", "bmi", "cholesterol", "glucose"]

knn = KNNImputer(n_neighbors=5)
data_knn[num_cols] = knn.fit_transform(data_knn[num_cols])

print("✅ Missing values after KNN:")
print(data_knn[num_cols].isnull().sum())

data_knn.head()

✅ Missing values after KNN:
age            0
bmi            0
cholesterol    0
glucose        0
dtype: int64


,patient_id,age,gender,region,bmi,blood_pressure,cholesterol,glucose,disease_risk
0,1,56.0,NaN,South,19.543048,117.996302,215.179282,105.768789,0
1,2,69.0,NaN,South,20.914993,136.287995,272.460193,95.553314,1
2,3,46.0,NaN,West,32.548857,88.486217,168.480974,73.645515,0
3,4,32.0,Male,South,27.010172,164.066293,166.839323,57.623927,1
4,5,60.0,Male,South,23.235360,106.932994,149.346380,47.684557,1


## 🔹 MICE Imputation

MICE (Multiple Imputation by Chained Equations) fills missing values by modeling each feature using other features.

It is more advanced and accurate than basic methods.

In [47]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

data_mice = data.copy()

mice = IterativeImputer()
data_mice[num_cols] = mice.fit_transform(data_mice[num_cols])

print("✅ Missing values after MICE:")
print(data_mice[num_cols].isnull().sum())

data_mice.head()

✅ Missing values after MICE:
age            0
bmi            0
cholesterol    0
glucose        0
dtype: int64


,patient_id,age,gender,region,bmi,blood_pressure,cholesterol,glucose,disease_risk
0,1,56.0,NaN,South,19.543048,117.996302,215.179282,105.768789,0
1,2,69.0,NaN,South,20.914993,136.287995,272.460193,95.553314,1
2,3,46.0,NaN,West,32.548857,88.486217,209.513464,73.645515,0
3,4,32.0,Male,South,27.010172,164.066293,166.839323,57.623927,1
4,5,60.0,Male,South,20.890199,106.932994,149.346380,47.684557,1


# Part B: Handling Outliers

## 🔹 Z-Score Method

Outliers are identified using standard deviation.

Values beyond ±3 standard deviations are considered outliers and removed.

In [48]:
from scipy.stats import zscore
import numpy as np

clean_data = data_mice.copy()

z = np.abs(zscore(clean_data[num_cols]))
data_z = clean_data[(z < 3).all(axis=1)]

print("Original Shape:", clean_data.shape)
print("After Z-score:", data_z.shape)

data_z.head()

Original Shape: (500, 9)
After Z-score: (481, 9)


,patient_id,age,gender,region,bmi,blood_pressure,cholesterol,glucose,disease_risk
0,1,56.0,NaN,South,19.543048,117.996302,215.179282,105.768789,0
1,2,69.0,NaN,South,20.914993,136.287995,272.460193,95.553314,1
2,3,46.0,NaN,West,32.548857,88.486217,209.513464,73.645515,0
3,4,32.0,Male,South,27.010172,164.066293,166.839323,57.623927,1
4,5,60.0,Male,South,20.890199,106.932994,149.346380,47.684557,1


## 🔹 IQR Method

The Interquartile Range (IQR) method detects outliers using quartiles.

Values outside 1.5 × IQR are treated as outliers.

In [49]:
Q1 = clean_data[num_cols].quantile(0.25)
Q3 = clean_data[num_cols].quantile(0.75)
IQR = Q3 - Q1

data_iqr = clean_data[~((clean_data[num_cols] < (Q1 - 1.5*IQR)) |
                        (clean_data[num_cols] > (Q3 + 1.5*IQR))).any(axis=1)]

print("After IQR:", data_iqr.shape)

data_iqr.head()

After IQR: (472, 9)


,patient_id,age,gender,region,bmi,blood_pressure,cholesterol,glucose,disease_risk
0,1,56.0,NaN,South,19.543048,117.996302,215.179282,105.768789,0
1,2,69.0,NaN,South,20.914993,136.287995,272.460193,95.553314,1
2,3,46.0,NaN,West,32.548857,88.486217,209.513464,73.645515,0
3,4,32.0,Male,South,27.010172,164.066293,166.839323,57.623927,1
4,5,60.0,Male,South,20.890199,106.932994,149.346380,47.684557,1


## 🔹 Percentile Capping

Extreme values are capped using percentile thresholds (1% and 99%).

This reduces the effect of extreme outliers without removing data.

In [50]:
lower = clean_data[num_cols].quantile(0.01)
upper = clean_data[num_cols].quantile(0.99)

data_pct = clean_data.copy()
data_pct[num_cols] = data_pct[num_cols].clip(lower, upper, axis=1)

print("After Percentile Capping:")
data_pct.describe()

After Percentile Capping:


,patient_id,age,bmi,blood_pressure,cholesterol,glucose,disease_risk
count,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000
mean,250.500000,49.430890,25.996152,123.565278,208.380686,103.437267,0.524000
std,144.481833,17.486322,6.571853,24.477274,54.690642,35.150525,0.499924
min,1.000000,18.000000,14.607278,76.675932,111.621576,30.062171,0.000000
25%,125.750000,36.000000,22.213822,109.474565,175.294914,80.784618,0.000000
50%,250.500000,49.241873,25.485166,120.947803,202.131833,102.098686,1.000000
75%,375.250000,64.000000,28.590863,130.912137,230.561635,120.189994,1.000000
max,500.000000,79.000000,55.384972,258.338829,441.844269,234.517989,1.000000


## 🔹 Winsorization

Outliers are replaced with boundary values instead of being removed.

This keeps dataset size unchanged while reducing extreme values.

In [51]:
from scipy.stats.mstats import winsorize

data_win = clean_data.copy()

for col in num_cols:
    data_win[col] = winsorize(data_win[col], limits=[0.05, 0.05])

print("After Winsorization:")
data_win.describe()

After Winsorization:


C:\Users\ASUS\AppData\Roaming\Python\Python311\site-packages\numpy\lib\function_base.py:4824: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(
C:\Users\ASUS\AppData\Roaming\Python\Python311\site-packages\numpy\lib\function_base.py:4824: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(
C:\Users\ASUS\AppData\Roaming\Python\Python311\site-packages\numpy\lib\function_base.py:4824: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(
C:\Users\ASUS\AppData\Roaming\Python\Python311\site-packages\numpy\lib\function_base.py:4824: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(
C:\Users\ASUS\AppData\Roaming\Python\Python311\site-packages\numpy\lib\function_base.py:4824: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(


,patient_id,age,bmi,blood_pressure,cholesterol,glucose,disease_risk
count,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000
mean,250.500000,49.404890,25.541004,123.565278,204.429424,102.110070,0.524000
std,144.481833,17.274387,4.812449,24.477274,39.427404,28.992466,0.499924
min,1.000000,20.000000,17.042977,76.675932,134.844393,51.293343,0.000000
25%,125.750000,36.000000,22.213822,109.474565,175.294914,80.784618,0.000000
50%,250.500000,49.241873,25.485166,120.947803,202.131833,102.098686,1.000000
75%,375.250000,64.000000,28.590863,130.912137,230.561635,120.189994,1.000000
max,500.000000,77.000000,35.869757,258.338829,287.499553,159.501587,1.000000


# Part C: Final Clean Dataset

## 🔹 Final Clean Dataset

The final dataset is created after applying:
- MICE (for missing values)
- IQR (for outlier removal)

This dataset is now ready for machine learning.

In [54]:
final_data = data_iqr.copy()
final_data.to_csv("cleaned_health_data.csv", index=False)

print("✅ Final dataset created")

final_data.head()

✅ Final dataset created


,patient_id,age,gender,region,bmi,blood_pressure,cholesterol,glucose,disease_risk
0,1,56.0,NaN,South,19.543048,117.996302,215.179282,105.768789,0
1,2,69.0,NaN,South,20.914993,136.287995,272.460193,95.553314,1
2,3,46.0,NaN,West,32.548857,88.486217,209.513464,73.645515,0
3,4,32.0,Male,South,27.010172,164.066293,166.839323,57.623927,1
4,5,60.0,Male,South,20.890199,106.932994,149.346380,47.684557,1


## 🔹 Before vs After Comparison

This section compares dataset shape and statistics before and after preprocessing.

In [53]:
print("Original Shape:", data.shape)
print("Final Shape:", final_data.shape)

print("\n🔹 Original Data Summary")
print(data.describe())

print("\n🔹 Cleaned Data Summary")
print(final_data.describe())

Original Shape: (500, 9)
Final Shape: (472, 9)

🔹 Original Data Summary
       patient_id         age         bmi  blood_pressure  cholesterol  \
count  500.000000  450.000000  450.000000      500.000000   450.000000   
mean   250.500000   49.448889   26.008593      123.565278   209.286945   
std    144.481833   18.433370    7.225784       24.477274    60.072028   
min      1.000000   18.000000    7.595956       76.675932    86.699217   
25%    125.750000   34.000000   21.786964      109.474565   174.030922   
50%    250.500000   50.000000   25.240848      120.947803   201.750579   
75%    375.250000   65.750000   28.766602      130.912137   233.041101   
max    500.000000   79.000000   66.463972      258.338829   556.855302   

          glucose  disease_risk  
count  450.000000    500.000000  
mean   103.498017      0.524000  
std     38.722890      0.499924  
min     -4.142897      0.000000  
25%     79.211563      0.000000  
50%    101.493817      1.000000  
75%    122.926564      

       patient_id         age         bmi  blood_pressure  cholesterol  \
count  472.000000  472.000000  472.000000      472.000000   472.000000   
mean   249.559322   49.160847   25.081022      119.807347   200.586275   
std    143.688315   17.568009    4.554018       14.731458    38.095993   
min      1.000000   18.000000   13.151393       76.675932   103.210190   
25%    126.750000   35.750000   22.079389      109.265147   174.318243   
50%    249.500000   49.217551   25.232279      119.972743   200.707788   
75%    373.250000   64.000000   28.146702      129.375226   225.761734   
max    500.000000   79.000000   37.849634      164.066293   309.234689   

          glucose  disease_risk  
count  472.000000    472.000000  
mean    99.858452      0.533898  
std     28.232689      0.499379  
min     23.248949      0.000000  
25%     80.457946      0.000000  
50%    101.040589      1.000000  
75%    117.219412      1.000000  
max    179.026661      1.000000  


## 📌 Final Observations

### ✔ Best Imputation Method
MICE performed best as it considers relationships between variables.

### 🔹 Why MICE?
MICE considers relationships between variables, making it more accurate than simple imputation.

### ✔ Best Outlier Method
IQR effectively removed extreme values.

### 🔹 Why IQR?
IQR is robust to skewed data and does not assume normal distribution.

### ✔ Improvements
- Missing values handled
- Outliers reduced
- Dataset ready for machine learning